<a href="https://colab.research.google.com/github/isaacadebayo/Agentic_projects/blob/main/Personal_assistant_MLFlow_GenAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#!pip install mlflow cryptography==43.0.3 --ignore-installed blinker

  Using cached mlflow-3.14.0-py3-none-any.whl.metadata (49 kB)
  Using cached cryptography-43.0.3-cp39-abi3-manylinux_2_28_x86_64.whl.metadata (5.4 kB)
  Using cached blinker-1.9.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached cffi-2.0.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (2.6 kB)
  Using cached mlflow_skinny-3.14.0-py3-none-any.whl.metadata (50 kB)
  Using cached mlflow_tracing-3.14.0-py3-none-any.whl.metadata (19 kB)
  Using cached flask_cors-6.0.5-py3-none-any.whl.metadata (5.4 kB)
  Using cached flask-3.1.3-py3-none-any.whl.metadata (3.2 kB)
  Using cached aiohttp-3.14.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (8.3 kB)
  Using cached alembic-1.18.5-py3-none-any.whl.metadata (7.2 kB)
  Using cached docker-7.1.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached graphene-3.4.3-py2.py3-none-any.whl.metadata (6.9 kB)
  Using cached gunicorn-26.0.0-py3-none-any.whl.metadata (5.4 kB)
  Using cached huey

Installed session restarted and code above commented out

In [1]:
!pip install pyngrok

In [2]:
import subprocess, time, requests, os, re
from pyngrok import ngrok
from google.colab import userdata

# ── 1. Kill everything ────────────────────────────────────────────
subprocess.run("pkill -f 'mlflow server'", shell=True)
subprocess.run("pkill -f proxy.py", shell=True)
subprocess.run("pkill -f cloudflared", shell=True)
time.sleep(3)

# ── 2. Start MLflow ───────────────────────────────────────────────
env = os.environ.copy()
env["GUNICORN_CMD_ARGS"] = "--forwarded-allow-ips='*'"

subprocess.Popen(
    ["mlflow", "server",
     "--backend-store-uri", "sqlite:////content/Agentic_projects/mlflow.db",
     "--serve-artifacts",
     "--host", "0.0.0.0",
     "--port", "5000",
     "--cors-allowed-origins", "https://ia-mlflow-dashboard.ngrok.app"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT,
    cwd="/content/Agentic_projects",
    env=env
)

print("Waiting for MLflow...")
for i in range(30):
    try:
        r = requests.get("http://localhost:5000/health", timeout=3)
        if r.status_code == 200:
            print(f"MLflow ready")
            break
    except Exception:
        pass
    time.sleep(2)

# ── 3. Write proxy (rewrites Host header to localhost:5000) ────────
proxy_code = """
import http.server
import http.client
import urllib.parse

class Proxy(http.server.BaseHTTPRequestHandler):
    def _proxy(self, body=None):
        parsed = urllib.parse.urlparse(f"http://localhost:5000{self.path}")

        conn = http.client.HTTPConnection("localhost", 5000, timeout=30)

        headers = dict(self.headers)
        headers["Host"] = "localhost:5000"
        headers["X-Forwarded-Host"] = "localhost:5000"
        headers["X-Forwarded-Proto"] = "http"
        headers.pop("Transfer-Encoding", None)
        headers.pop("transfer-encoding", None)

        try:
            conn.request(self.command, parsed.path + (f"?{parsed.query}" if parsed.query else ""), body=body, headers=headers)
            resp = conn.getresponse()

            self.send_response(resp.status)

            for k, v in resp.getheaders():
                if k.lower() not in ("transfer-encoding", "connection", "keep-alive"):
                    self.send_header(k, v)
            self.send_header("Access-Control-Allow-Origin", "*")
            self.send_header("Access-Control-Allow-Headers", "*")
            self.end_headers()

            # Stream response in chunks — fixes chart data loading
            while True:
                chunk = resp.read(8192)
                if not chunk:
                    break
                self.wfile.write(chunk)
            self.wfile.flush()

        except Exception as e:
            self.send_response(502)
            self.end_headers()
            self.wfile.write(str(e).encode())
        finally:
            conn.close()

    def do_GET(self):     self._proxy()
    def do_POST(self):
        length = int(self.headers.get("Content-Length", 0))
        self._proxy(self.rfile.read(length) if length else None)
    def do_PUT(self):
        length = int(self.headers.get("Content-Length", 0))
        self._proxy(self.rfile.read(length) if length else None)
    def do_DELETE(self):  self._proxy()
    def do_OPTIONS(self):
        self.send_response(200)
        self.send_header("Access-Control-Allow-Origin", "*")
        self.send_header("Access-Control-Allow-Methods", "GET, POST, PUT, DELETE, OPTIONS")
        self.send_header("Access-Control-Allow-Headers", "*")
        self.end_headers()
    def log_message(self, *args): pass

server = http.server.HTTPServer(('0.0.0.0', 5001), Proxy)
server.serve_forever()
"""

with open("proxy.py", "w") as f:
    f.write(proxy_code)

subprocess.Popen(["python", "proxy.py"],
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print("Proxy started on port 5001")
time.sleep(3)

# ── 4. Connect ngrok to proxy on port 5001 ────────────────────────
ngrok.set_auth_token(userdata.get('NGROK_AUTH_TOKEN'))

# Replace with your custom domain
public_url = ngrok.connect(5001, domain="ia-mlflow-dashboard.ngrok.app")
print(f"\nMLflow UI: {public_url}\n")

Waiting for MLflow...
MLflow ready
Proxy started on port 5001

MLflow UI: NgrokTunnel: "https://ia-mlflow-dashboard.ngrok.app" -> "http://localhost:5001"



In [3]:
import mlflow
import mlflow.langchain
import mlflow.openai
import os
import openai
import json

# Set the MLflow tracking URI to connect to the server started by ngrok
# The ngrok setup usually exposes localhost:5000
mlflow.set_tracking_uri("http://localhost:5000")

# Set an experiment name for better organization in the MLflow UI
mlflow.set_experiment("Multimodal Reasoning Personal Assistant Chatbot Interactions")
mlflow.openai.autolog()

print("MLflow tracking setup complete.")

MLflow tracking setup complete.


In [4]:
# Aggressively uninstall all potentially conflicting packages
!pip uninstall -y langchain langchain-core langchain-community langchain-openai langchain-huggingface pydantic mlflow gradio pyngrok cryptography

# Install a consistent langchain 0.2.x set (core pinned so the pydantic_v1 shim exists)
!pip install --upgrade -q langchain==0.2.16 langchain-core==0.2.43 langchain-text-splitters==0.2.4

# Stable Pydantic v2 known to work with Langchain 0.2.x
!pip install --upgrade -q pydantic==2.7.4

# Other core Langchain components (versions matched to 0.2.x)
!pip install --upgrade -q langchain-community==0.2.11 langchain-openai==0.1.15 langchain-huggingface==0.0.3

# Install remaining dependencies
!pip install --upgrade -q faiss-cpu pypdf bs4 openai google-genai langgraph==0.2.0 "numpy<2" scipy==1.13.1 sentence-transformers edge-tts mlflow pyngrok cryptography==43.0.3 gradio

Found existing installation: langchain 0.2.16
Uninstalling langchain-0.2.16:
  Successfully uninstalled langchain-0.2.16
Found existing installation: langchain-core 0.2.43
Uninstalling langchain-core-0.2.43:
  Successfully uninstalled langchain-core-0.2.43
Found existing installation: langchain-community 0.2.11
Uninstalling langchain-community-0.2.11:
  Successfully uninstalled langchain-community-0.2.11
Found existing installation: langchain-openai 0.1.15
Uninstalling langchain-openai-0.1.15:
  Successfully uninstalled langchain-openai-0.1.15
Found existing installation: langchain-huggingface 0.0.3
Uninstalling langchain-huggingface-0.0.3:
  Successfully uninstalled langchain-huggingface-0.0.3
Found existing installation: pydantic 2.13.4
Uninstalling pydantic-2.13.4:
  Successfully uninstalled pydantic-2.13.4
Found existing installation: mlflow 3.14.0
Uninstalling mlflow-3.14.0:
  Successfully uninstalled mlflow-3.14.0
Found existing installation: gradio 6.19.0
Uninstalling gradio-6.1

RESTART SESSION, RUNTIME (T4) AND COMMENT OUT

In [4]:
import gradio as gr
import getpass
import os
import bs4
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from openai import OpenAI
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from typing_extensions import List, TypedDict
from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.prompts import ChatPromptTemplate
from google import genai
from google.genai import types
from google.colab import drive

In [5]:
drive.mount('/content/drive')

dataset = '/content/drive/MyDrive/convo6000.csv'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
import pandas as pd
df = pd.read_csv(dataset)
df.head()

,data
0,i usually enjoy the weekends but not this one
1,we tried to scare them away but they almost at...
2,i was nervous for no reason
3,have you tried to look into other providers to...
4,i do not need anything else in life


In [7]:
from langchain_core.documents import Document # Import Document class

loader_df = pd.read_csv(dataset)
docs = []
for index, row in loader_df.iterrows():
    docs.append(Document(page_content=row['data']))

# Join the page content from the loaded documents into a single string for test_chatbot
personal_assistant_chatbot = "\n\n".join([doc.page_content for doc in docs])

In [8]:
personal_assistant_chatbot

'i usually enjoy the weekends but not this one\n\nwe tried to scare them away but they almost attacked me\n\ni was nervous for no reason\n\nhave you tried to look into other providers to see if there is a better deal available\n\ni do not need anything else in life\n\nyou can always get yourself another dog\n\ni was the one who was being seduced as the married man\n\ni have to call my landlord about being late on the rent\n\ni cannot elieve my daughter is starting highschool\n\nthe game was full of intrigue\n\nuniversity is starting up soon and i am nervous\n\nnice i love watching cute dogs doing cool things\n\nwhy what is happening on monday\n\ni have not practiced enough\n\nsome of my daughters classmates in prek have been crying since the first day of school\n\nyes but it took us a few minutes to do so because we were so shocked\n\ni did one of those water coasters the other day\n\ni just ate four donuts myself\n\ni rehearsed all night for my speech\n\ni worked one job where i was a

In [9]:
#!pip install sentence-transformers

In [10]:
# 2. Setting up Embeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

print("Setup complete. Your chatbot is ready for testing, Let's Go Isaac!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Setup complete. Your chatbot is ready for testing, Let's Go Isaac!


If the huggig face embedding does not work there is an error between numpy and scipy (version mismatch)

In [11]:
# FIX: langgraph 0.2.0 has no 'langgraph.types' module (added in later 0.2.x). Upgrade langgraph while staying compatible with langchain-core 0.2.43
# Then Runtime > Restart session before re-running the import cell
!pip install --upgrade "langgraph>=0.2.60,<0.3"

In [12]:
from google.colab import userdata

In [13]:
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

In [14]:
llm = ChatOpenAI(model="gpt-5-turbo", temperature=0.8, max_tokens=250)

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [15]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)

In [16]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=50, chunk_overlap=10)
all_splits = text_splitter.split_documents(docs)

In [17]:
# Create FAISS index from chunks
db = FAISS.from_documents(all_splits, embeddings)

[Trace(trace_id=tr-eb3d05a6fae75a6da4b9dc3cba2f8d64), Trace(trace_id=tr-d39393362eafea09a950750df0555850), Trace(trace_id=tr-270aaa4a80749d3c061b01293edd378e), Trace(trace_id=tr-7f89eef68a64aacc50898c0dc980fef6), Trace(trace_id=tr-38b7c755bf05c553b65281aa0a2a9362), Trace(trace_id=tr-e8717475bc20834ca2f47434ea5d141b), Trace(trace_id=tr-92c395b16c8883696abf7d359e6b5693), Trace(trace_id=tr-7dcaeb24d32b673beee46fa7ec31c753), Trace(trace_id=tr-ae6ac470a22643c68c9b848d46e35cc6)]

In [18]:
#!pip install "uvicorn==0.29"

In [19]:
#!pip install --upgrade uvicorn asyncio-contextmanager

In [20]:
'''import openai
import os
import uuid
import asyncio
import nest_asyncio
nest_asyncio.apply()
import edge_tts
import base64
import tempfile
import numpy as np
from io import BytesIO
from PIL import Image as PILImage
from typing import List, Annotated, TypedDict, Sequence
from langchain_core.messages import (
    HumanMessage, AIMessage, SystemMessage, BaseMessage, ToolMessage, RemoveMessage,
)
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages          # NEW: reducer that supports RemoveMessage
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver        # NEW: persistence for memory + interrupts
from langgraph.types import interrupt, Command             # NEW: human-in-the-loop primitives
import gradio as gr

# ─────────────────────────────────────────────
# Token / memory budget constants
# ─────────────────────────────────────────────
MAX_RAG_CHARS     = 800
MAX_RAG_CHUNKS    = 2
MAX_AGENT_TOKENS  = 512
KEEP_RECENT_MSGS  = 6      # NEW: messages kept verbatim; older ones get summarized
SUMMARY_MAX_TOK   = 220    # NEW: token budget for the rolling memory summary

# Tools that require explicit human approval before they run.
# generate_image costs money / has side effects, so it's gated. Add more names to gate them too.
REVIEW_TOOLS = {"generate_image"}   # NEW


# --- compatibility shim for uvicorn 0.49 + nest_asyncio ---
_current_run = asyncio.run
def _run_no_loop_factory(coro, **kwargs):
    kwargs.pop("loop_factory", None)
    return _current_run(coro, **kwargs)
asyncio.run = _run_no_loop_factory

# ─────────────────────────────────────────────
# 1. Helper — encode image file to base64
# ─────────────────────────────────────────────
def encode_image_to_base64(image_path: str) -> str:
    with open(image_path, "rb") as f:
        raw = f.read()
    ext  = image_path.rsplit(".", 1)[-1].lower()
    mime = {"jpg": "image/jpeg", "jpeg": "image/jpeg",
            "png": "image/png",  "gif":  "image/gif",
            "webp": "image/webp"}.get(ext, "image/png")
    b64 = base64.b64encode(raw).decode("utf-8")
    return f"data:{mime};base64,{b64}"

# ─────────────────────────────────────────────
# 2. @tool — RAG search
# ─────────────────────────────────────────────
@tool
def rag_search(question: str) -> str:
    """Search the financial document database and return a concise answer."""
    try:
        retrieved_docs = db.similarity_search(question, k=MAX_RAG_CHUNKS)
        chunks  = [doc.page_content[:MAX_RAG_CHARS]
                   for doc in retrieved_docs
                   if hasattr(doc, "page_content") and doc.page_content.strip()]
        context = "\n---\n".join(chunks)[:MAX_RAG_CHARS * MAX_RAG_CHUNKS]
        rag_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2, max_tokens=250)
        msgs    = [
            SystemMessage(content=(
                "You are a financial analyst. Answer using ONLY the context below. "
                "Be concise (2-3 sentences). Say so if context lacks the answer."
            )),
            HumanMessage(content=f"Context:\n{context}\n\nQuestion: {question}"),
        ]
        return rag_llm.invoke(msgs).content
    except Exception as e:
        return f"RAG search error: {e}"

# ─────────────────────────────────────────────
# 3. @tool — Analyze uploaded/camera image (GPT-4o vision)
# ─────────────────────────────────────────────
@tool
def analyze_image(image_path: str) -> str:
    """
    Analyze a user-uploaded or webcam-captured image using GPT-5 vision.
    Provide analysis of the image.
    Use this whenever the user provides an image via upload or camera.
    """
    try:
        data_uri   = encode_image_to_base64(image_path)
        vision_llm = ChatOpenAI(model="gpt-4o", max_tokens=400)
        msgs = [HumanMessage(content=[
            {"type": "text",      "text": (
                "You are a conversational assistant. Analyze this image. "
                "Tell me your opinion if you have no improvement let me know. "
                "Otherwise describe what you think."
            )},
            {"type": "image_url", "image_url": {"url": data_uri}},
        ])]
        return vision_llm.invoke(msgs).content
    except Exception as e:
        return f"Image analysis error: {e}"

# ─────────────────────────────────────────────
# 4. @tool — Image generation (gpt-image-1)  [gated by human review]
# ─────────────────────────────────────────────
@tool
def generate_image(visual_prompt: str) -> str:
    """
    Generate an image with gpt-image-1. Call ONLY when the user explicitly asks
    for a new image or visualization to be created.
    Returns IMAGE_GENERATED::<uri> — include verbatim in reply.
    """
    print(f"[gpt-image-1] Generating: {visual_prompt}")
    try:
        response = client.images.generate(
            model="gpt-image-1", prompt=visual_prompt, size="1024x1024", n=1,
        )
        if not response or not response.data:
            return "Image generation failed: empty response."
        img_data = response.data[0]
        if hasattr(img_data, "b64_json") and img_data.b64_json:
            return f"IMAGE_GENERATED::data:image/png;base64,{img_data.b64_json}"
        elif hasattr(img_data, "url") and img_data.url:
            return f"IMAGE_GENERATED::{img_data.url}"
        return "Image generation failed: no data."
    except Exception as e:
        return f"Image generation failed: {e}"

agent_tools = [rag_search, analyze_image, generate_image]

# ─────────────────────────────────────────────
# 5. Agent state
#    NOTE: switched reducer from operator.add -> add_messages so the memory
#    node can delete old messages with RemoveMessage. Added `summary` for
#    long-term memory and `review_decision` for the HITL routing flag.
# ─────────────────────────────────────────────
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]
    summary: str
    review_decision: str
    reasoning: str

SYSTEM_PROMPT = (
    "You are my assistant created by Isaac Adebayo "
    "Use rag_search to look up information when prompted "
    "Use analyze_image when the user uploads or captures an image — the image_path is embedded in their message as [IMAGE_PATH: ...]. "
    "Use generate_image ONLY when the user explicitly asks for a new image to be created. "
    "When generate_image returns IMAGE_GENERATED::<uri>, include that exact string verbatim in your reply. "
    "Keep text answers concise (3-4 sentences max)."
)

agent_llm   = ChatOpenAI(model="gpt-4o-mini", temperature=0.7, max_tokens=MAX_AGENT_TOKENS)
llm_w_tools = agent_llm.bind_tools(agent_tools)

# ─────────────────────────────────────────────
# 6a. MEMORY AGENT — rolling summary node
#     Keeps the last KEEP_RECENT_MSGS messages verbatim and folds everything
#     older into `summary`, deleting them from the message list. Only cuts on a
#     clean turn boundary (a HumanMessage) so tool_call/tool_response pairs are
#     never split (OpenAI rejects orphaned tool calls).
# ─────────────────────────────────────────────
def _safe_cut(messages: Sequence[BaseMessage], target: int) -> int:
    """Return the smallest index >= target that starts a fresh user turn, else 0."""
    for i in range(target, len(messages)):
        if isinstance(messages[i], HumanMessage):
            return i
    return 0

# ─────────────────────────────────────────────
# 6a-bis. REASON node — explicit deliberation before the agent acts.
# Produces a short private analysis, NOT a user-facing answer. The agent
# node reads it to guide the reply. Key behavior: revise beliefs when the
# user gives a credible DIRECT observation, but don't invent facts and
# don't blindly capitulate.
# ─────────────────────────────────────────────
REASONING_MAX_TOK = 300
reasoning_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2, max_tokens=REASONING_MAX_TOK)

def reason_node(state: AgentState) -> dict:
    convo = [m for m in state["messages"] if not isinstance(m, SystemMessage)]
    sys = SystemMessage(content=(
        "You are the private reasoning module for an assistant. Think step by step "
        "about the user's LATEST message. Do NOT answer the user — produce a short "
        "internal analysis (<=120 words):\n"
        "1. What is the user asking or asserting?\n"
        "2. What do I actually know vs. not know? Do I have direct evidence?\n"
        "3. If the user contradicts a claim I made: they can physically observe their "
        "surroundings and I cannot, so defer to a credible DIRECT observation and update "
        "my belief. But do not accept implausible claims, and do not invent facts. Note "
        "that 'not X' does not by itself imply a specific 'Y' (e.g. 'not raining' could be "
        "sunny OR cloudy — flag the ambiguity instead of guessing).\n"
        "4. Do I need a tool (rag_search / analyze_image) to ground the answer?"
    ))
    try:
        thought = reasoning_llm.invoke([sys] + convo).content
    except Exception as e:
        thought = ""
        print(f"[reason] failed: {e}")
    return {"reasoning": thought}

def memory_node(state: AgentState) -> dict:
    messages = list(state["messages"])
    if len(messages) <= KEEP_RECENT_MSGS:
        return {}

    cut = _safe_cut(messages, len(messages) - KEEP_RECENT_MSGS)
    if cut < 2:
        return {}  # nothing safe to summarize yet

    old      = messages[:cut]
    prior    = state.get("summary", "")

    # Build a compact transcript of the messages we're about to fold away.
    lines = []
    for m in old:
        if isinstance(m, HumanMessage):
            txt = m.content if isinstance(m.content, str) else "[image/multimodal message]"
            lines.append(f"User: {txt}")
        elif isinstance(m, AIMessage) and isinstance(m.content, str) and m.content.strip():
            lines.append(f"Assistant: {m.content}")
        elif isinstance(m, ToolMessage):
            snippet = (m.content or "")[:200]
            lines.append(f"ToolResult: {snippet}")
    transcript = "\n".join(lines).strip()
    if not transcript:
        # Nothing worth summarizing (e.g. only tool noise) — just drop it.
        return {"messages": [RemoveMessage(id=m.id) for m in old if getattr(m, "id", None)]}

    try:
        summary_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3, max_tokens=SUMMARY_MAX_TOK)
        sys = SystemMessage(content=(
            "You maintain a running memory of a persoanl conversational assistant. "
            "Merge the existing summary with the new exchange into a single concise summary "
            "(<=150 words). Preserve user goals, tickers, figures, and decisions."
        ))
        human = HumanMessage(content=(
            f"Existing summary:\n{prior or '(none)'}\n\nNew exchange:\n{transcript}\n\nUpdated summary:"
        ))
        new_summary = summary_llm.invoke([sys, human]).content
    except Exception as e:
        # If summarization fails, keep the old summary and still trim to bound context.
        new_summary = prior
        print(f"[memory] summarization failed: {e}")

    removals = [RemoveMessage(id=m.id) for m in old if getattr(m, "id", None)]
    return {"summary": new_summary, "messages": removals}

# ─────────────────────────────────────────────
# 6b. Agent node — injects the memory summary into the system prompt
# ─────────────────────────────────────────────
def agent_node(state: AgentState) -> dict:
    convo  = [m for m in state["messages"] if not isinstance(m, SystemMessage)]
    system = SYSTEM_PROMPT
    if state.get("summary"):
        system += f"\n\n[Memory — earlier conversation summary]\n{state['summary']}"
    if state.get("reasoning"):
        system += ("\n\n[Internal reasoning — let this guide your reply; "
                   "do NOT repeat it verbatim to the user]\n" + state["reasoning"])
    full = [SystemMessage(content=system)] + convo
    return {"messages": [llm_w_tools.invoke(full)]}

def should_continue(state: AgentState) -> str:
    last = state["messages"][-1]
    return "review" if getattr(last, "tool_calls", None) else "end"

# ─────────────────────────────────────────────
# 6c. HUMAN-IN-THE-LOOP — review node
#     Passes through cheap/safe tools automatically. For REVIEW_TOOLS it calls
#     interrupt(), which pauses the graph and surfaces the pending call to the
#     UI. Execution resumes when the Gradio side calls invoke(Command(resume=…)).
# ─────────────────────────────────────────────
def human_review_node(state: AgentState) -> dict:
    last = state["messages"][-1]
    tool_calls = getattr(last, "tool_calls", []) or []
    needs_review = [tc for tc in tool_calls if tc["name"] in REVIEW_TOOLS]

    if not needs_review:
        return {"review_decision": "approve"}     # nothing gated -> straight to tools

    # Pause here. `value` is what the UI receives; the return value is whatever
    # the UI passes back via Command(resume=...).
    decision = interrupt({
        "type": "tool_approval",
        "tool_calls": [{"name": tc["name"], "args": tc["args"]} for tc in needs_review],
    })
    action = decision.get("action", "reject") if isinstance(decision, dict) else str(decision)

    if action == "approve":
        return {"review_decision": "approve"}

    # Rejected: satisfy the "every tool_call needs a tool response" invariant by
    # injecting a ToolMessage for each pending call, then hand back to the agent.
    rejects = [
        ToolMessage(
            tool_call_id=tc["id"],
            content=f"❌ User declined to run `{tc['name']}`. Do not retry; explain or ask what they'd prefer.",
        )
        for tc in tool_calls
    ]
    return {"messages": rejects, "review_decision": "reject"}

def route_after_review(state: AgentState) -> str:
    return "tools" if state.get("review_decision") == "approve" else "agent"

# ─────────────────────────────────────────────
# 7. Build LangGraph  (memory -> agent -> [review -> tools] loop)
# ─────────────────────────────────────────────
tool_node = ToolNode(agent_tools)
workflow  = StateGraph(AgentState)
workflow.add_node("memory",       memory_node)
workflow.add_node("agent",        agent_node)
workflow.add_node("human_review", human_review_node)
workflow.add_node("tools",        tool_node)

workflow.set_entry_point("memory")
workflow.add_edge("memory", "agent")
workflow.add_conditional_edges("agent", should_continue, {"review": "human_review", "end": END})
workflow.add_conditional_edges("human_review", route_after_review, {"tools": "tools", "agent": "agent"})
workflow.add_edge("tools", "agent")
workflow.add_node("reason", reason_node)          # NEW
workflow.add_edge("memory", "reason")             # NEW
workflow.add_edge("reason",  "agent")             # NEW (was memory -> agent)

# A checkpointer is REQUIRED for interrupt()/resume and gives us persistent memory
# per thread_id. MemorySaver is in-process (fine for a demo / single Space).
# For durable, multi-restart memory swap in SqliteSaver or a Postgres checkpointer.
checkpointer = MemorySaver()
agent_graph  = workflow.compile(checkpointer=checkpointer)
print("✅ LangGraph agentic workflow (memory + human-in-the-loop) compiled successfully!")

# ─────────────────────────────────────────────
# 8. Reply parsing + result rendering helpers
# ─────────────────────────────────────────────
def parse_agent_reply(raw: str):
    marker = "IMAGE_GENERATED::"
    if marker in raw:
        parts    = raw.split(marker, 1)
        text     = parts[0].strip()
        uri      = parts[1].strip().split()[0]
        trailing = parts[1].strip()[len(uri):].strip()
        return (text + " " + trailing).strip(), uri
    return raw, None

def _extract_final(result: dict):
    for m in reversed(result["messages"]):
        if isinstance(m, AIMessage) and not getattr(m, "tool_calls", None):
            raw = m.content if isinstance(m.content, str) else str(m.content)
            return parse_agent_reply(raw)
    return "I could not generate a response. Please try again.", None

def _finalize(result: dict, history: list):
    """Update chat history from a graph result; reveal approval row if paused."""
    interrupts = result.get("__interrupt__")
    if interrupts:
        payload = interrupts[0].value
        calls   = payload.get("tool_calls", [])
        lines   = "\n".join(f"• `{c['name']}` → {c['args']}" for c in calls)
        history.append({
            "role": "assistant",
            "content": f"🔔 **Approval required** before running:\n{lines}\n\n"
                       f"Use **Approve** / **Reject** below.",
        })
        return history, gr.update(visible=True)

    text, uri = _extract_final(result)
    if text:
        history.append({"role": "assistant", "content": text})
    if uri:
        history.append({"role": "assistant", "content": gr.Image(value=uri)})
    return history, gr.update(visible=False)

# ─────────────────────────────────────────────
# 9. Predict (text + optional image) — checkpointer holds prior turns,
#    so we only send the NEW message each call.
# ─────────────────────────────────────────────
def predict_agentic(message: str, image_input, history: list, session_id):
    if history is None:
        history = []
    if not session_id:
        session_id = str(uuid.uuid4())
    config = {"configurable": {"thread_id": session_id}}

    try:
        if image_input is not None:
            if isinstance(image_input, np.ndarray):
                img = PILImage.fromarray(image_input.astype("uint8"))
                tmp = tempfile.NamedTemporaryFile(suffix=".png", delete=False)
                img.save(tmp.name)
                image_path = tmp.name
            else:
                image_path = str(image_input)
            user_text = message.strip() if message and message.strip() else "Please analyze this image."
            content   = f"{user_text}\n[IMAGE_PATH: {image_path}]"
        else:
            user_text = message or ""
            content   = user_text

        history.append({"role": "user", "content": user_text})
        result = agent_graph.invoke({"messages": [HumanMessage(content=content)]}, config)
        history, approval_update = _finalize(result, history)
        return history, "", None, session_id, approval_update

    except Exception as e:
        err = f"Agent error: {e}"
        print(err)
        history.append({"role": "assistant", "content": err})
        return history, "", None, session_id, gr.update(visible=False)

# ─────────────────────────────────────────────
# 9b. Resume after a human approval / rejection
# ─────────────────────────────────────────────
def resume_agentic(action: str, history: list, session_id):
    if history is None:
        history = []
    if not session_id:
        return history, gr.update(visible=False), session_id
    config = {"configurable": {"thread_id": session_id}}
    try:
        result = agent_graph.invoke(Command(resume={"action": action}), config)
        history, approval_update = _finalize(result, history)
        return history, approval_update, session_id
    except Exception as e:
        err = f"Resume error: {e}"
        print(err)
        history.append({"role": "assistant", "content": err})
        return history, gr.update(visible=False), session_id

# ─────────────────────────────────────────────
# 10. TTS / voice
# ─────────────────────────────────────────────
async def _tts(text: str) -> str:
    communicate = edge_tts.Communicate(text, "en-US-AndrewNeural")
    path = "response_agent.mp3"
    await communicate.save(path)
    return path

def transcribe_audio(audio_path: str) -> str:
    if not os.environ.get("OPENAI_API_KEY"):
        return "Error: OPENAI_API_KEY not set."
    _client = openai.OpenAI()
    with open(audio_path, "rb") as f:
        return _client.audio.transcriptions.create(model="whisper-1", file=f).text

def voice_agent_handler(audio_path, history, session_id):
    if history is None:
        history = []
    if audio_path is None:
        return history, None, "No audio received.", session_id, gr.update(visible=False)
    try:
        user_text = transcribe_audio(audio_path)
        if user_text.startswith("Error:"):
            return history, None, user_text, session_id, gr.update(visible=False)

        history, _, _, session_id, approval_update = predict_agentic(user_text, None, history, session_id)

        # If the graph paused for approval, don't speak — wait for the buttons.
        config  = {"configurable": {"thread_id": session_id}}
        paused  = bool(agent_graph.get_state(config).next)
        if paused:
            return history, None, "Awaiting your approval…", session_id, approval_update

        bot_text = ""
        for item in reversed(history):
            if isinstance(item, dict) and item.get("role") == "assistant":
                c = item.get("content", "")
                if isinstance(c, str) and not c.startswith("data:image"):
                    bot_text = c
                    break
        audio_out = None
        if bot_text:
            loop = asyncio.get_event_loop()
            audio_out = asyncio.get_event_loop().run_until_complete(_tts(bot_text))
        return history, audio_out, "Done!", session_id, approval_update

    except Exception as e:
        err = f"Voice error: {e}"
        history.append({"role": "assistant", "content": err})
        return history, None, err, session_id, gr.update(visible=False)

# ─────────────────────────────────────────────
# 11. Gradio app
# ─────────────────────────────────────────────
with gr.Blocks() as agentic_app:
    gr.Markdown("## 🤖 Agentic Multimodal RAG Assistant — Memory + Human-in-the-Loop")
    gr.Markdown("*memory → agent → (human review) → tools → agent … with a rolling summary and approval gate on image generation.*")

    session_state = gr.State(None)   # per-session thread_id for the checkpointer

    chatbot = gr.Chatbot(label="Chat History", height=450)

    with gr.Row():
        msg      = gr.Textbox(placeholder="Ask a question, or upload/snap an image below...", scale=3, label="Your message")
        audio_in = gr.Audio(sources="microphone", type="filepath", scale=1, label="🎤 Voice")

    image_input = gr.Image(
        sources=["upload", "webcam"],
        type="numpy",
        label="📷 Upload Image or Use Camera",
        height=220,
    )

    with gr.Row():
        submit_btn = gr.Button("Send", variant="primary", scale=2)
        clear_btn  = gr.Button("Clear", scale=1)

    # Human-in-the-loop approval controls (hidden until the graph pauses)
    with gr.Row(visible=False) as approval_row:
        approve_btn = gr.Button("✅ Approve", variant="primary")
        reject_btn  = gr.Button("🚫 Reject", variant="stop")

    with gr.Row():
        audio_out  = gr.Audio(label="🔊 Voice Response", autoplay=True)
        status_box = gr.Textbox(label="Status", interactive=False)

    submit_btn.click(predict_agentic, [msg, image_input, chatbot, session_state],
                     [chatbot, msg, image_input, session_state, approval_row])
    msg.submit(predict_agentic,       [msg, image_input, chatbot, session_state],
                     [chatbot, msg, image_input, session_state, approval_row])

    approve_btn.click(lambda h, s: resume_agentic("approve", h, s),
                      [chatbot, session_state], [chatbot, approval_row, session_state])
    reject_btn.click(lambda h, s: resume_agentic("reject", h, s),
                     [chatbot, session_state], [chatbot, approval_row, session_state])

    # Clear also starts a fresh memory thread by resetting session_state to None.
    clear_btn.click(lambda: ([], "", None, None, gr.update(visible=False)),
                    outputs=[chatbot, msg, image_input, session_state, approval_row])

    audio_in.stop_recording(voice_agent_handler, [audio_in, chatbot, session_state],
                            [chatbot, audio_out, status_box, session_state, approval_row])

agentic_app.launch(share=True)'''

'import openai\nimport os\nimport uuid\nimport asyncio\nimport nest_asyncio\nnest_asyncio.apply()\nimport edge_tts\nimport base64\nimport tempfile\nimport numpy as np\nfrom io import BytesIO\nfrom PIL import Image as PILImage\nfrom typing import List, Annotated, TypedDict, Sequence\nfrom langchain_core.messages import (\n    HumanMessage, AIMessage, SystemMessage, BaseMessage, ToolMessage, RemoveMessage,\n)\nfrom langchain_core.tools import tool\nfrom langchain_openai import ChatOpenAI\nfrom langgraph.graph import StateGraph, END\nfrom langgraph.graph.message import add_messages          # NEW: reducer that supports RemoveMessage\nfrom langgraph.prebuilt import ToolNode\nfrom langgraph.checkpoint.memory import MemorySaver        # NEW: persistence for memory + interrupts\nfrom langgraph.types import interrupt, Command             # NEW: human-in-the-loop primitives\nimport gradio as gr\n\n# ─────────────────────────────────────────────\n# Token / memory budget constants\n# ────────────

In [21]:
#!pip install --upgrade "langgraph>=0.2.60,<0.3"

In [22]:
import openai
import os
import uuid
import asyncio
import nest_asyncio
nest_asyncio.apply()
import edge_tts
import base64
import tempfile
import numpy as np
from io import BytesIO
from PIL import Image as PILImage
from typing import List, Annotated, TypedDict, Sequence
from langchain_core.messages import (
    HumanMessage, AIMessage, SystemMessage, BaseMessage, ToolMessage, RemoveMessage,
)
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages          # reducer that supports RemoveMessage
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver        # persistence for memory + interrupts
from langgraph.types import interrupt, Command             # human-in-the-loop primitives
import gradio as gr


# ─────────────────────────────────────────────
# Token / memory budget constants
# ─────────────────────────────────────────────
MAX_RAG_CHARS     = 800
MAX_RAG_CHUNKS    = 2
MAX_AGENT_TOKENS  = 512
KEEP_RECENT_MSGS  = 6      # messages kept verbatim; older ones get summarized
SUMMARY_MAX_TOK   = 220    # token budget for the rolling memory summary

# Tools that require explicit human approval before they run.
# generate_image costs money / has side effects, so it's gated. Add more names to gate them too.
REVIEW_TOOLS = {"generate_image"}

# NOTE: The uvicorn 0.29 "compatibility shim" that monkeypatched asyncio.run has been
# removed. You are on uvicorn 0.49 (no patch needed), and the stacked patch was a likely
# cause of the RecursionError / "coroutine 'Server.serve' was never awaited" warning.


# ─────────────────────────────────────────────
# 1. Helper — encode image file to base64
# ─────────────────────────────────────────────
def encode_image_to_base64(image_path: str) -> str:
    with open(image_path, "rb") as f:
        raw = f.read()
    ext  = image_path.rsplit(".", 1)[-1].lower()
    mime = {"jpg": "image/jpeg", "jpeg": "image/jpeg",
            "png": "image/png",  "gif":  "image/gif",
            "webp": "image/webp"}.get(ext, "image/png")
    b64 = base64.b64encode(raw).decode("utf-8")
    return f"data:{mime};base64,{b64}"


# ─────────────────────────────────────────────
# 2. @tool — RAG search
# ─────────────────────────────────────────────
@tool
def rag_search(question: str) -> str:
    """Search the financial document database and return a concise answer."""
    try:
        retrieved_docs = db.similarity_search(question, k=MAX_RAG_CHUNKS)
        chunks  = [doc.page_content[:MAX_RAG_CHARS]
                   for doc in retrieved_docs
                   if hasattr(doc, "page_content") and doc.page_content.strip()]
        context = "\n---\n".join(chunks)[:MAX_RAG_CHARS * MAX_RAG_CHUNKS]
        rag_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2, max_tokens=250)
        msgs    = [
            SystemMessage(content=(
                "You are a financial analyst. Answer using ONLY the context below. "
                "Be concise (2-3 sentences). Say so if context lacks the answer."
            )),
            HumanMessage(content=f"Context:\n{context}\n\nQuestion: {question}"),
        ]
        return rag_llm.invoke(msgs).content
    except Exception as e:
        return f"RAG search error: {e}"


# ─────────────────────────────────────────────
# 3. @tool — Analyze uploaded/camera image (GPT-4o vision)
# ─────────────────────────────────────────────
@tool
def analyze_image(image_path: str) -> str:
    """
    Analyze a user-uploaded or webcam-captured image using GPT-4o vision.
    Provide analysis of the image.
    Use this whenever the user provides an image via upload or camera.
    """
    try:
        data_uri   = encode_image_to_base64(image_path)
        vision_llm = ChatOpenAI(model="gpt-4o", max_tokens=400)
        msgs = [HumanMessage(content=[
            {"type": "text",      "text": (
                "You are a conversational assistant. Analyze this image. "
                "Tell me your opinion if you have no improvement let me know. "
                "Otherwise describe what you think."
            )},
            {"type": "image_url", "image_url": {"url": data_uri}},
        ])]
        return vision_llm.invoke(msgs).content
    except Exception as e:
        return f"Image analysis error: {e}"


# ─────────────────────────────────────────────
# 4. @tool — Image generation (gpt-image-1)  [gated by human review]
# ─────────────────────────────────────────────
@tool
def generate_image(visual_prompt: str) -> str:
    """
    Generate an image with gpt-image-1. Call ONLY when the user explicitly asks
    for a new image or visualization to be created.
    Returns IMAGE_GENERATED::<uri> — include verbatim in reply.
    """
    print(f"[gpt-image-1] Generating: {visual_prompt}")
    try:
        response = client.images.generate(
            model="gpt-image-1", prompt=visual_prompt, size="1024x1024", n=1,
        )
        if not response or not response.data:
            return "Image generation failed: empty response."
        img_data = response.data[0]
        if hasattr(img_data, "b64_json") and img_data.b64_json:
            return f"IMAGE_GENERATED::data:image/png;base64,{img_data.b64_json}"
        elif hasattr(img_data, "url") and img_data.url:
            return f"IMAGE_GENERATED::{img_data.url}"
        return "Image generation failed: no data."
    except Exception as e:
        return f"Image generation failed: {e}"


agent_tools = [rag_search, analyze_image, generate_image]


# ─────────────────────────────────────────────
# 5. Agent state
# ─────────────────────────────────────────────
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]
    summary: str
    review_decision: str
    reasoning: str


SYSTEM_PROMPT = (
    "You are my assistant created by Isaac Adebayo "
    "Use rag_search to look up information when prompted "
    "Use analyze_image when the user uploads or captures an image — the image_path is embedded in their message as [IMAGE_PATH: ...]. "
    "Use generate_image ONLY when the user explicitly asks for a new image to be created. "
    "When generate_image returns IMAGE_GENERATED::<uri>, include that exact string verbatim in your reply. "
    "Keep text answers concise (3-4 sentences max)."
)

agent_llm   = ChatOpenAI(model="gpt-4o-mini", temperature=0.7, max_tokens=MAX_AGENT_TOKENS)
llm_w_tools = agent_llm.bind_tools(agent_tools)


# ─────────────────────────────────────────────
# 6a. MEMORY AGENT — rolling summary node
# ─────────────────────────────────────────────
def _safe_cut(messages: Sequence[BaseMessage], target: int) -> int:
    """Return the smallest index >= target that starts a fresh user turn, else 0."""
    for i in range(target, len(messages)):
        if isinstance(messages[i], HumanMessage):
            return i
    return 0


# ─────────────────────────────────────────────
# 6a-bis. REASON node — explicit deliberation before the agent acts.
# ─────────────────────────────────────────────
REASONING_MAX_TOK = 300
reasoning_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2, max_tokens=REASONING_MAX_TOK)


def reason_node(state: AgentState) -> dict:
    convo = [m for m in state["messages"] if not isinstance(m, SystemMessage)]
    sys = SystemMessage(content=(
        "You are the private reasoning module for an assistant. Think step by step "
        "about the user's LATEST message. Do NOT answer the user — produce a short "
        "internal analysis (<=120 words):\n"
        "1. What is the user asking or asserting?\n"
        "2. What do I actually know vs. not know? Do I have direct evidence?\n"
        "3. If the user contradicts a claim I made: they can physically observe their "
        "surroundings and I cannot, so defer to a credible DIRECT observation and update "
        "my belief. But do not accept implausible claims, and do not invent facts. Note "
        "that 'not X' does not by itself imply a specific 'Y' (e.g. 'not raining' could be "
        "sunny OR cloudy — flag the ambiguity instead of guessing).\n"
        "4. Do I need a tool (rag_search / analyze_image) to ground the answer?"
    ))
    try:
        thought = reasoning_llm.invoke([sys] + convo).content
    except Exception as e:
        thought = ""
        print(f"[reason] failed: {e}")
    return {"reasoning": thought}


def memory_node(state: AgentState) -> dict:
    messages = list(state["messages"])
    if len(messages) <= KEEP_RECENT_MSGS:
        return {}

    cut = _safe_cut(messages, len(messages) - KEEP_RECENT_MSGS)
    if cut < 2:
        return {}  # nothing safe to summarize yet

    old   = messages[:cut]
    prior = state.get("summary", "")

    # Build a compact transcript of the messages we're about to fold away.
    lines = []
    for m in old:
        if isinstance(m, HumanMessage):
            txt = m.content if isinstance(m.content, str) else "[image/multimodal message]"
            lines.append(f"User: {txt}")
        elif isinstance(m, AIMessage) and isinstance(m.content, str) and m.content.strip():
            lines.append(f"Assistant: {m.content}")
        elif isinstance(m, ToolMessage):
            snippet = (m.content or "")[:200]
            lines.append(f"ToolResult: {snippet}")
    transcript = "\n".join(lines).strip()
    if not transcript:
        # Nothing worth summarizing (e.g. only tool noise) — just drop it.
        return {"messages": [RemoveMessage(id=m.id) for m in old if getattr(m, "id", None)]}

    try:
        summary_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3, max_tokens=SUMMARY_MAX_TOK)
        sys = SystemMessage(content=(
            "You maintain a running memory of a personal conversational assistant. "
            "Merge the existing summary with the new exchange into a single concise summary "
            "(<=150 words). Preserve user goals, tickers, figures, and decisions."
        ))
        human = HumanMessage(content=(
            f"Existing summary:\n{prior or '(none)'}\n\nNew exchange:\n{transcript}\n\nUpdated summary:"
        ))
        new_summary = summary_llm.invoke([sys, human]).content
    except Exception as e:
        # If summarization fails, keep the old summary and still trim to bound context.
        new_summary = prior
        print(f"[memory] summarization failed: {e}")

    removals = [RemoveMessage(id=m.id) for m in old if getattr(m, "id", None)]
    return {"summary": new_summary, "messages": removals}


# ─────────────────────────────────────────────
# 6b. Agent node — injects the memory summary into the system prompt
# ─────────────────────────────────────────────
def agent_node(state: AgentState) -> dict:
    convo  = [m for m in state["messages"] if not isinstance(m, SystemMessage)]
    system = SYSTEM_PROMPT
    if state.get("summary"):
        system += f"\n\n[Memory — earlier conversation summary]\n{state['summary']}"
    if state.get("reasoning"):
        system += ("\n\n[Internal reasoning — let this guide your reply; "
                   "do NOT repeat it verbatim to the user]\n" + state["reasoning"])
    full = [SystemMessage(content=system)] + convo
    return {"messages": [llm_w_tools.invoke(full)]}


def should_continue(state: AgentState) -> str:
    last = state["messages"][-1]
    return "review" if getattr(last, "tool_calls", None) else "end"


# ─────────────────────────────────────────────
# 6c. HUMAN-IN-THE-LOOP — review node
# ─────────────────────────────────────────────
def human_review_node(state: AgentState) -> dict:
    last = state["messages"][-1]
    tool_calls = getattr(last, "tool_calls", []) or []
    needs_review = [tc for tc in tool_calls if tc["name"] in REVIEW_TOOLS]

    if not needs_review:
        return {"review_decision": "approve"}     # nothing gated -> straight to tools

    # Pause here. `value` is what the UI receives; the return value is whatever
    # the UI passes back via Command(resume=...).
    decision = interrupt({
        "type": "tool_approval",
        "tool_calls": [{"name": tc["name"], "args": tc["args"]} for tc in needs_review],
    })
    action = decision.get("action", "reject") if isinstance(decision, dict) else str(decision)

    if action == "approve":
        return {"review_decision": "approve"}

    # Rejected: satisfy the "every tool_call needs a tool response" invariant by
    # injecting a ToolMessage for each pending call, then hand back to the agent.
    rejects = [
        ToolMessage(
            tool_call_id=tc["id"],
            content=f"❌ User declined to run `{tc['name']}`. Do not retry; explain or ask what they'd prefer.",
        )
        for tc in tool_calls
    ]
    return {"messages": rejects, "review_decision": "reject"}


def route_after_review(state: AgentState) -> str:
    return "tools" if state.get("review_decision") == "approve" else "agent"


# ─────────────────────────────────────────────
# 7. Build LangGraph  (memory -> reason -> agent -> [review -> tools] loop)
# ─────────────────────────────────────────────
tool_node = ToolNode(agent_tools)
workflow  = StateGraph(AgentState)
workflow.add_node("memory",       memory_node)
workflow.add_node("reason",       reason_node)
workflow.add_node("agent",        agent_node)
workflow.add_node("human_review", human_review_node)
workflow.add_node("tools",        tool_node)

workflow.set_entry_point("memory")
workflow.add_edge("memory", "reason")     # FIXED: memory -> reason (removed duplicate memory -> agent)
workflow.add_edge("reason", "agent")      # reason -> agent
workflow.add_conditional_edges("agent", should_continue, {"review": "human_review", "end": END})
workflow.add_conditional_edges("human_review", route_after_review, {"tools": "tools", "agent": "agent"})
workflow.add_edge("tools", "agent")

# A checkpointer is REQUIRED for interrupt()/resume and gives us persistent memory
# per thread_id. MemorySaver is in-process (fine for a demo / single Space).
checkpointer = MemorySaver()
agent_graph  = workflow.compile(checkpointer=checkpointer)
print("✅ LangGraph agentic workflow (memory + human-in-the-loop) compiled successfully!")


# ─────────────────────────────────────────────
# 8. Reply parsing + result rendering helpers
# ─────────────────────────────────────────────
def parse_agent_reply(raw: str):
    marker = "IMAGE_GENERATED::"
    if marker in raw:
        parts    = raw.split(marker, 1)
        text     = parts[0].strip()
        uri      = parts[1].strip().split()[0]
        trailing = parts[1].strip()[len(uri):].strip()
        return (text + " " + trailing).strip(), uri
    return raw, None


def _extract_final(result: dict):
    for m in reversed(result["messages"]):
        if isinstance(m, AIMessage) and not getattr(m, "tool_calls", None):
            raw = m.content if isinstance(m.content, str) else str(m.content)
            return parse_agent_reply(raw)
    return "I could not generate a response. Please try again.", None


def _finalize(result: dict, history: list):
    """Update chat history from a graph result; reveal approval row if paused."""
    interrupts = result.get("__interrupt__")
    if interrupts:
        payload = interrupts[0].value
        calls   = payload.get("tool_calls", [])
        lines   = "\n".join(f"• `{c['name']}` → {c['args']}" for c in calls)
        history.append({
            "role": "assistant",
            "content": f"🔔 **Approval required** before running:\n{lines}\n\n"
                       f"Use **Approve** / **Reject** below.",
        })
        return history, gr.update(visible=True)

    text, uri = _extract_final(result)
    if text:
        history.append({"role": "assistant", "content": text})
    if uri:
        history.append({"role": "assistant", "content": gr.Image(value=uri)})
    return history, gr.update(visible=False)


# ─────────────────────────────────────────────
# 9. Predict (text + optional image)
# ─────────────────────────────────────────────
def predict_agentic(message: str, image_input, history: list, session_id):
    if history is None:
        history = []
    if not session_id:
        session_id = str(uuid.uuid4())
    config = {"configurable": {"thread_id": session_id}}

    try:
        if image_input is not None:
            if isinstance(image_input, np.ndarray):
                img = PILImage.fromarray(image_input.astype("uint8"))
                tmp = tempfile.NamedTemporaryFile(suffix=".png", delete=False)
                img.save(tmp.name)
                image_path = tmp.name
            else:
                image_path = str(image_input)
            user_text = message.strip() if message and message.strip() else "Please analyze this image."
            content   = f"{user_text}\n[IMAGE_PATH: {image_path}]"
        else:
            user_text = message or ""
            content   = user_text

        history.append({"role": "user", "content": user_text})
        result = agent_graph.invoke({"messages": [HumanMessage(content=content)]}, config)
        history, approval_update = _finalize(result, history)
        return history, "", None, session_id, approval_update

    except Exception as e:
        err = f"Agent error: {e}"
        print(err)
        history.append({"role": "assistant", "content": err})
        return history, "", None, session_id, gr.update(visible=False)


# ─────────────────────────────────────────────
# 9b. Resume after a human approval / rejection
# ─────────────────────────────────────────────
def resume_agentic(action: str, history: list, session_id):
    if history is None:
        history = []
    if not session_id:
        return history, gr.update(visible=False), session_id
    config = {"configurable": {"thread_id": session_id}}
    try:
        result = agent_graph.invoke(Command(resume={"action": action}), config)
        history, approval_update = _finalize(result, history)
        return history, approval_update, session_id
    except Exception as e:
        err = f"Resume error: {e}"
        print(err)
        history.append({"role": "assistant", "content": err})
        return history, gr.update(visible=False), session_id


# ─────────────────────────────────────────────
# 10. TTS / voice
# ─────────────────────────────────────────────
async def _tts(text: str) -> str:
    communicate = edge_tts.Communicate(text, "en-US-AndrewNeural")
    path = "response_agent.mp3"
    await communicate.save(path)
    return path


def transcribe_audio(audio_path: str) -> str:
    if not os.environ.get("OPENAI_API_KEY"):
        return "Error: OPENAI_API_KEY not set."
    _client = openai.OpenAI()
    with open(audio_path, "rb") as f:
        return _client.audio.transcriptions.create(model="whisper-1", file=f).text


def voice_agent_handler(audio_path, history, session_id):
    if history is None:
        history = []
    if audio_path is None:
        return history, None, "No audio received.", session_id, gr.update(visible=False)
    try:
        user_text = transcribe_audio(audio_path)
        if user_text.startswith("Error:"):
            return history, None, user_text, session_id, gr.update(visible=False)

        # FIXED: proper tuple unpacking (was garbled with empty slots)
        history, _, _, session_id, approval_update = predict_agentic(user_text, None, history, session_id)

        # If the graph paused for approval, don't speak — wait for the buttons.
        config = {"configurable": {"thread_id": session_id}}
        paused = bool(agent_graph.get_state(config).next)
        if paused:
            return history, None, "Awaiting your approval…", session_id, approval_update

        bot_text = ""
        for item in reversed(history):
            if isinstance(item, dict) and item.get("role") == "assistant":
                c = item.get("content", "")
                if isinstance(c, str) and not c.startswith("data:image"):
                    bot_text = c
                    break

        audio_out = None
        if bot_text:
            loop = asyncio.get_event_loop()            # FIXED: reuse the loop, no double call
            audio_out = loop.run_until_complete(_tts(bot_text))
        return history, audio_out, "Done!", session_id, approval_update

    except Exception as e:
        err = f"Voice error: {e}"
        history.append({"role": "assistant", "content": err})
        return history, None, err, session_id, gr.update(visible=False)


# ─────────────────────────────────────────────
# 11. Gradio app
# ─────────────────────────────────────────────
with gr.Blocks() as agentic_app:
    gr.Markdown("## 🤖 Agentic Multimodal RAG Assistant — Memory + Human-in-the-Loop")
    gr.Markdown("*memory → reason → agent → (human review) → tools → agent … with a rolling summary and approval gate on image generation.*")

    session_state = gr.State(None)   # per-session thread_id for the checkpointer

    chatbot = gr.Chatbot(label="Chat History", height=450)

    with gr.Row():
        msg      = gr.Textbox(placeholder="Ask a question, or upload/snap an image below...", scale=3, label="Your message")
        audio_in = gr.Audio(sources="microphone", type="filepath", scale=1, label="🎤 Voice")

    image_input = gr.Image(
        sources=["upload", "webcam"],
        type="numpy",
        label="📷 Upload Image or Use Camera",
        height=220,
    )

    with gr.Row():
        submit_btn = gr.Button("Send", variant="primary", scale=2)
        clear_btn  = gr.Button("Clear", scale=1)

    # Human-in-the-loop approval controls (hidden until the graph pauses)
    with gr.Row(visible=False) as approval_row:
        approve_btn = gr.Button("✅ Approve", variant="primary")
        reject_btn  = gr.Button("🚫 Reject", variant="stop")

    with gr.Row():
        audio_out  = gr.Audio(label="🔊 Voice Response", autoplay=True)
        status_box = gr.Textbox(label="Status", interactive=False)

    submit_btn.click(predict_agentic, [msg, image_input, chatbot, session_state],
                     [chatbot, msg, image_input, session_state, approval_row])
    msg.submit(predict_agentic,       [msg, image_input, chatbot, session_state],
                     [chatbot, msg, image_input, session_state, approval_row])

    approve_btn.click(lambda h, s: resume_agentic("approve", h, s),
                      [chatbot, session_state], [chatbot, approval_row, session_state])
    reject_btn.click(lambda h, s: resume_agentic("reject", h, s),
                     [chatbot, session_state], [chatbot, approval_row, session_state])

    # Clear also starts a fresh memory thread by resetting session_state to None.
    clear_btn.click(lambda: ([], "", None, None, gr.update(visible=False)),
                    outputs=[chatbot, msg, image_input, session_state, approval_row])

    audio_in.stop_recording(voice_agent_handler, [audio_in, chatbot, session_state],
                            [chatbot, audio_out, status_box, session_state, approval_row])

agentic_app.launch(share=True)

✅ LangGraph agentic workflow (memory + human-in-the-loop) compiled successfully!
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f11fe3035905381969.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [23]:
import uvicorn
import sys
print(f"Python: {sys.version}")
print(f"uvicorn: {uvicorn.__version__}")

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
uvicorn: 0.50.0


In [24]:
#!pip install nest_asyncio

Uses uvicorn 0.49 and not 0.29